# Predicting Sales from Advertising Spend

**Portfolio version of a university machine-learning project**

This project compares four supervised-learning approaches for predicting product sales from advertising expenditure:

1. Multiple linear regression
2. Polynomial regression
3. Regression tree
4. Support vector regression (SVR)

The goal is not only to fit each model, but to compare them on the **same held-out test set** using RMSE, MAE, and \(R^2\).

The dataset contains advertising spending on TV, radio, and newspapers together with observed sales.

## 1. Setup

The notebook uses R. Install the required packages once if they are not already available:

```r
install.packages(c("tidyverse", "boot", "tree", "e1071"))
```

In [ ]:
library(tidyverse)
library(boot)
library(tree)
library(e1071)

theme_set(theme_minimal(base_size = 12))

## 2. Load and inspect the data

In [ ]:
data_path <- if (file.exists("data/Advertising.csv")) {
  "data/Advertising.csv"
} else if (file.exists("Advertising.csv")) {
  "Advertising.csv"
} else {
  stop("Advertising.csv was not found. Open the notebook from the repository root or place the file in data/.")
}

advertising <- read.csv(data_path) |>
  as_tibble() |>
  select(-matches("^Unnamed|^X$"))

glimpse(advertising)

tibble(
  observations = nrow(advertising),
  missing_values = sum(is.na(advertising)),
  duplicated_rows = sum(duplicated(advertising))
)


The response variable is **sales**. The predictors are advertising expenditures for **TV**, **radio**, and **newspaper**.

Before modeling, it is useful to inspect the marginal relationships between each advertising channel and sales.

In [ ]:
advertising |>
  pivot_longer(c(TV, radio, newspaper),
               names_to = "channel",
               values_to = "spend") |>
  ggplot(aes(x = spend, y = sales)) +
  geom_point(alpha = 0.65) +
  geom_smooth(method = "lm", se = FALSE) +
  facet_wrap(~ channel, scales = "free_x") +
  labs(
    title = "Sales versus advertising spend",
    x = "Advertising spend",
    y = "Sales"
  )

## 3. Train/test split

In [ ]:
set.seed(41599)

train_index <- sample(
  seq_len(nrow(advertising)),
  size = floor(0.70 * nrow(advertising)),
  replace = FALSE
)

train <- advertising[train_index, ]
test  <- advertising[-train_index, ]

cat("Training observations:", nrow(train), "\n")
cat("Test observations:", nrow(test), "\n")

All models below are fitted **only on the training data**. The test set is reserved for final out-of-sample comparison.

In [ ]:
regression_metrics <- function(actual, predicted) {
  tibble(
    RMSE = sqrt(mean((actual - predicted)^2)),
    MAE  = mean(abs(actual - predicted)),
    R2   = 1 - sum((actual - predicted)^2) /
                 sum((actual - mean(actual))^2)
  )
}

## 4. Model 1 — Multiple linear regression

In [ ]:
linear_model <- lm(
  sales ~ TV + radio + newspaper,
  data = train
)

summary(linear_model)

linear_test_pred <- predict(linear_model, newdata = test)
linear_metrics <- regression_metrics(test$sales, linear_test_pred)
linear_metrics

The coefficient table is useful for interpreting the linear model, but predictive performance should be judged using the held-out test set rather than by fitting a separate model to the test data.

## 5. Model 2 — Polynomial regression

In [ ]:
set.seed(41599)

degrees <- 1:5

cv_results <- map_dfr(degrees, function(d) {
  model <- glm(
    sales ~ poly(TV, d, raw = TRUE) +
            poly(radio, d, raw = TRUE),
    data = train
  )

  cv_mse <- cv.glm(train, model, K = 10)$delta[1]

  tibble(degree = d, CV_MSE = cv_mse)
})

cv_results

ggplot(cv_results, aes(x = degree, y = CV_MSE)) +
  geom_line() +
  geom_point(size = 2) +
  scale_x_continuous(breaks = degrees) +
  labs(
    title = "10-fold cross-validation for polynomial degree",
    x = "Polynomial degree",
    y = "Cross-validated MSE"
  )

In [ ]:
best_degree <- cv_results$degree[which.min(cv_results$CV_MSE)]
cat("Selected polynomial degree:", best_degree, "\n")

poly_formula <- as.formula(
  paste0(
    "sales ~ poly(TV, ", best_degree, ", raw = TRUE) + ",
    "poly(radio, ", best_degree, ", raw = TRUE)"
  )
)

polynomial_model <- lm(poly_formula, data = train)

poly_test_pred <- predict(polynomial_model, newdata = test)
polynomial_metrics <- regression_metrics(test$sales, poly_test_pred)
polynomial_metrics

Newspaper advertising is omitted from the polynomial model because the earlier linear analysis provides little evidence that it improves the prediction of sales. The polynomial degree is selected by cross-validation instead of being chosen manually.

## 6. Model 3 — Regression tree

In [ ]:
set.seed(41599)

full_tree <- tree(
  sales ~ TV + radio + newspaper,
  data = train,
  mindev = 0,
  minsize = 5
)

tree_cv <- cv.tree(full_tree, K = 10)

best_tree_size <- tree_cv$size[which.min(tree_cv$dev)]
cat("Selected tree size:", best_tree_size, "\n")

pruned_tree <- prune.tree(full_tree, best = best_tree_size)

plot(pruned_tree)
text(pruned_tree, pretty = 0)

tree_test_pred <- predict(pruned_tree, newdata = test)
tree_metrics <- regression_metrics(test$sales, tree_test_pred)
tree_metrics

## 7. Model 4 — Support vector regression

In [ ]:
set.seed(41599)

svr_tuning <- tune.svm(
  sales ~ TV + radio + newspaper,
  data = train,
  kernel = "polynomial",
  degree = c(2, 3, 4),
  cost = c(0.1, 1, 10),
  gamma = c(0.01, 0.1, 1)
)

svr_tuning$best.parameters

svr_model <- svr_tuning$best.model

svr_test_pred <- predict(svr_model, newdata = test)
svr_metrics <- regression_metrics(test$sales, svr_test_pred)
svr_metrics

Because **sales is continuous**, the comparable support-vector model is **support vector regression (SVR)**. A separate high/low-sales SVM classifier would answer a different question and is therefore not included in the main model comparison.

## 8. Out-of-sample model comparison

In [ ]:
comparison <- bind_rows(
  linear_metrics     |> mutate(Model = "Linear regression"),
  polynomial_metrics |> mutate(Model = "Polynomial regression"),
  tree_metrics       |> mutate(Model = "Regression tree"),
  svr_metrics        |> mutate(Model = "Support vector regression")
) |>
  select(Model, everything()) |>
  arrange(RMSE)

comparison

In [ ]:
prediction_plot_data <- bind_rows(
  tibble(Actual = test$sales, Predicted = linear_test_pred,
         Model = "Linear regression"),
  tibble(Actual = test$sales, Predicted = poly_test_pred,
         Model = "Polynomial regression"),
  tibble(Actual = test$sales, Predicted = tree_test_pred,
         Model = "Regression tree"),
  tibble(Actual = test$sales, Predicted = svr_test_pred,
         Model = "Support vector regression")
)

ggplot(prediction_plot_data, aes(x = Actual, y = Predicted)) +
  geom_point(alpha = 0.7) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
  facet_wrap(~ Model) +
  coord_equal() +
  labs(
    title = "Actual versus predicted sales on the test set",
    x = "Actual sales",
    y = "Predicted sales"
  )

## 9. Residual diagnostics for the best test-set model

In [ ]:
best_model_name <- comparison$Model[1]

best_predictions <- switch(
  best_model_name,
  "Linear regression" = linear_test_pred,
  "Polynomial regression" = poly_test_pred,
  "Regression tree" = tree_test_pred,
  "Support vector regression" = svr_test_pred
)

diagnostics <- tibble(
  Predicted = best_predictions,
  Residual = test$sales - best_predictions
)

ggplot(diagnostics, aes(x = Predicted, y = Residual)) +
  geom_point(alpha = 0.7) +
  geom_hline(yintercept = 0, linetype = "dashed") +
  labs(
    title = paste("Residual plot —", best_model_name),
    x = "Predicted sales",
    y = "Residual"
  )

## 10. Takeaways

This project illustrates a complete supervised-learning workflow:

- exploratory data analysis;
- a reproducible train/test split;
- cross-validation for model complexity and hyperparameters;
- comparison of four regression approaches on the same held-out observations;
- model evaluation with RMSE, MAE, and \(R^2\);
- residual diagnostics for the best-performing model.

A key methodological lesson is that the test set should be used for **evaluation, not model fitting**, and that adjusted \(R^2\) is not a meaningful common metric for comparing linear models with trees and support vector machines.

## Data source

The notebook uses the commonly circulated **Advertising** dataset containing TV, radio, newspaper, and sales variables. The data file included here is the version used for the original university project.